In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Load the "Full" weights (CNN + MLP) from your Script B run
# Make sure you point to the file that has BOTH or the separate ones you saved
CNN_WEIGHTS = os.path.join(BASE_PATH, "HMVCL_Encoder_CNN.weights.h5")
MLP_WEIGHTS = os.path.join(BASE_PATH, "HMVCL_Encoder_MLP.weights.h5")

LABEL_PERCENTAGE = 0.20
LATENT_DIM = 128

# --- Definitions ---
def get_cnn_encoder(input_shape): # Same CNN def
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu')(h) # Projection
    return Model(inputs, [h, z])

def get_mlp_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu')(h)
    return Model(inputs, [h, z])

def main():
    # 1. Load Data
    print("Loading Data...")
    X_payload = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_payload = X_payload.reshape(X_payload.shape[0], -1, 1)

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename']
    stats_cols = [c for c in df.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]
    X_stats = StandardScaler().fit_transform(df[stats_cols].values.astype('float32'))

    # 2. Extract Latent Features (DEEP FUSION)
    print("Extracting Latent Features for Deep Fusion...")

    cnn = get_cnn_encoder(X_payload.shape[1:])
    cnn.load_weights(CNN_WEIGHTS)
    cnn_h = Model(inputs=cnn.input, outputs=cnn.outputs[0]).predict(X_payload, verbose=0)

    mlp = get_mlp_encoder(X_stats.shape[1])
    mlp.load_weights(MLP_WEIGHTS)
    mlp_h = Model(inputs=mlp.input, outputs=mlp.outputs[0]).predict(X_stats, verbose=0)

    # FUSE LATENT + LATENT (Deep Fusion)
    X_final = np.concatenate([cnn_h, mlp_h], axis=1)

    # 3. Train XGBoost
    print("Training Deep Fusion Baseline (XGBoost on Latent+Latent)...")

    # Reconstruct Labels (Simplified for brevity)
    app_col = next(c for c in df.columns if c.endswith('application'))
    cat_col = next(c for c in df.columns if c.endswith('category'))

    # ... (Insert your label reconstruction loop here) ...
    # For now assuming you have y_app, y_cat lists...
    # (Copy the label reconstruction block from previous scripts)

    # Example App Task Run
    # (Insert XGBoost training block here)

if __name__ == "__main__":
    main()

Loading Data...
Extracting Latent Features for Deep Fusion...


ValueError: A total of 1 objects could not be loaded. Example error message for object <Dense name=representation, built=True>:

The shape of the target variable and the shape of the target value in `variable.assign(value)` must match. variable.shape=(15616, 128), Received: value.shape=(2048, 128). Target variable: <Variable path=representation/kernel, shape=(15616, 128), dtype=float32, value=[[ 0.01419735 -0.01582436  0.00508595 ... -0.00137625 -0.01630562
   0.01598051]
 [ 0.00167275  0.01188419  0.00919147 ... -0.01130118 -0.0120388
   0.01662456]
 [ 0.01701014  0.012179    0.00649673 ... -0.0149281   0.0140851
  -0.00663572]
 ...
 [ 0.00400983 -0.0145124   0.0182134  ...  0.0016924   0.01679293
  -0.00774377]
 [ 0.00957484  0.011558    0.00864012 ... -0.01650147 -0.00253389
   0.00374486]
 [-0.00691595 -0.01935841  0.00365365 ... -0.00112321 -0.00794047
  -0.00779947]]>

List of objects that could not be loaded:
[<Dense name=representation, built=True>]

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Weights (From your Standard Pre-training)
# Update exact filenames if they differ on your drive
CNN_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_CNN.weights.h5")
MLP_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_MLP.weights.h5")

# Config
LABEL_PERCENTAGE = 0.05
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Architecture Definitions (Must match Pre-training) ---

# A. CNN Encoder (View 1)
def get_cnn_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((input_dim, 1))(inputs)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_Encoder")

# B. MLP Encoder (View 2)
def get_mlp_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="MLP_Encoder")

# --- 2. Data Loading Helper ---
def load_and_prep_data():
    print("--- Loading Merged CSV ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # View 1: Alpha
    cols_v1 = sorted([c for c in df.columns if c.startswith('alpha_alpha_pp_')],
                     key=lambda x: int(x.split('_')[-1]))

    # View 2: Stats (Beta + Gamma + FFT)
    exclude = ['application', 'category', 'binary_type', 'filename']
    cols_v2 = [c for c in df.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
               and not any(x in c for x in exclude)]

    print(f"Columns Found -> V1 (Payload): {len(cols_v1)}, V2 (Stats): {len(cols_v2)}")

    X1 = df[cols_v1].values.astype('float32')
    X2 = df[cols_v2].values.astype('float32')

    # Normalize Stats (Essential for MLP)
    print("Normalizing Features...")
    X2 = StandardScaler().fit_transform(X2)

    return df, X1, X2

# --- 3. Trainer Helper (No SMOTE) ---
def train_task_head_no_smote(X_subset, y_subset, task_name):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # Use sample weights for balance
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")
    return y_test, y_pred

# --- 4. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X1, X2 = load_and_prep_data()
    if df_raw is None: return

    # 2. Reconstruct Labels
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # 3. Feature Fusion (Dual Latent Representations)
    print("\n--- Generating Latent Representations (CNN + MLP) ---")

    # Define Models
    cnn_enc = get_cnn_encoder(X1.shape[1])
    mlp_enc = get_mlp_encoder(X2.shape[1])

    print(f"Loading Weights:\n - {CNN_WEIGHTS_PATH}\n - {MLP_WEIGHTS_PATH}")
    try:
        cnn_enc.load_weights(CNN_WEIGHTS_PATH)
        mlp_enc.load_weights(MLP_WEIGHTS_PATH)
    except Exception as e:
        print(f"FATAL: Could not load weights.\n{e}")
        return

    # Extract Latent Features (Cut off projection head)
    # We use index 0 -> h (representation)
    cnn_ext = Model(inputs=cnn_enc.input, outputs=cnn_enc.outputs[0])
    mlp_ext = Model(inputs=mlp_enc.input, outputs=mlp_enc.outputs[0])

    h_cnn = cnn_ext.predict(X1, batch_size=128, verbose=0)
    h_mlp = mlp_ext.predict(X2, batch_size=128, verbose=0)

    print(f"Latent Shapes: CNN={h_cnn.shape}, MLP={h_mlp.shape}")

    # FUSE: [Latent Payload (128) + Latent Stats (128)]
    # This creates a semantic vector of size 256
    X_final = np.concatenate([h_cnn, h_mlp], axis=1)
    print(f"Fused Super-Vector Shape: {X_final.shape}")

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task_head_no_smote(X_final, y_bin, "Binary Task")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))

    # ==========================================
    # EXPERIMENT 2: VPN CATEGORY
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY")
    print("="*40)

    mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final[mask]
    y_cat_raw = df_labels.loc[mask, 'Category']

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)
    y_test, y_pred = train_task_head_no_smote(X_cat, y_cat, "VPN Category")
    print(classification_report(y_test, y_pred, target_names=le_cat.classes_))

    # ==========================================
    # EXPERIMENT 3: VPN TOP APPS
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN TOP 6 APPS")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask = df_labels['Application'].isin(target_apps)
    X_app = X_final[mask]
    y_app_raw = df_labels.loc[mask, 'Application']

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)
    y_test, y_pred = train_task_head_no_smote(X_app, y_app, "VPN Top Apps")
    print(classification_report(y_test, y_pred, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV ---
Columns Found -> V1 (Payload): 128, V2 (Stats): 137
Normalizing Features...
Reconstructing Labels...

--- Generating Latent Representations (CNN + MLP) ---
Loading Weights:
 - /content/drive/MyDrive/1 Skripsi/27jan/HMVCL_Encoder_CNN.weights.h5
 - /content/drive/MyDrive/1 Skripsi/27jan/HMVCL_Encoder_MLP.weights.h5
Latent Shapes: CNN=(12555, 128), MLP=(12555, 128)
Fused Super-Vector Shape: (12555, 256)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 256)
    Train: 627 | Test: 11928
    >>> Binary Task Weighted F1: 0.8958
              precision    recall  f1-score   support

     Non-VPN       0.93      0.93      0.93      9314
         VPN       0.76      0.77      0.76      2614

    accuracy                           0.90     11928
   macro avg       0.85      0.85      0.85     11928
weighted avg       0.90      0.90      0.90     11928


 EXPERIMENT 2: VPN CATEGORY

>>> Starting Task: VPN Category
    Data Shape: (

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Weights (From your Standard Pre-training)
# Update exact filenames if they differ on your drive
CNN_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_CNN.weights.h5")
MLP_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_MLP.weights.h5")

# Config
LABEL_PERCENTAGE = 0.10
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Architecture Definitions (Must match Pre-training) ---

# A. CNN Encoder (View 1)
def get_cnn_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((input_dim, 1))(inputs)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_Encoder")

# B. MLP Encoder (View 2)
def get_mlp_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="MLP_Encoder")

# --- 2. Data Loading Helper ---
def load_and_prep_data():
    print("--- Loading Merged CSV ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # View 1: Alpha
    cols_v1 = sorted([c for c in df.columns if c.startswith('alpha_alpha_pp_')],
                     key=lambda x: int(x.split('_')[-1]))

    # View 2: Stats (Beta + Gamma + FFT)
    exclude = ['application', 'category', 'binary_type', 'filename']
    cols_v2 = [c for c in df.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
               and not any(x in c for x in exclude)]

    print(f"Columns Found -> V1 (Payload): {len(cols_v1)}, V2 (Stats): {len(cols_v2)}")

    X1 = df[cols_v1].values.astype('float32')
    X2 = df[cols_v2].values.astype('float32')

    # Normalize Stats (Essential for MLP)
    print("Normalizing Features...")
    X2 = StandardScaler().fit_transform(X2)

    return df, X1, X2

# --- 3. Trainer Helper (No SMOTE) ---
def train_task_head_no_smote(X_subset, y_subset, task_name):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # Use sample weights for balance
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")
    return y_test, y_pred

# --- 4. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X1, X2 = load_and_prep_data()
    if df_raw is None: return

    # 2. Reconstruct Labels
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # 3. Feature Fusion (Dual Latent Representations)
    print("\n--- Generating Latent Representations (CNN + MLP) ---")

    # Define Models
    cnn_enc = get_cnn_encoder(X1.shape[1])
    mlp_enc = get_mlp_encoder(X2.shape[1])

    print(f"Loading Weights:\n - {CNN_WEIGHTS_PATH}\n - {MLP_WEIGHTS_PATH}")
    try:
        cnn_enc.load_weights(CNN_WEIGHTS_PATH)
        mlp_enc.load_weights(MLP_WEIGHTS_PATH)
    except Exception as e:
        print(f"FATAL: Could not load weights.\n{e}")
        return

    # Extract Latent Features (Cut off projection head)
    # We use index 0 -> h (representation)
    cnn_ext = Model(inputs=cnn_enc.input, outputs=cnn_enc.outputs[0])
    mlp_ext = Model(inputs=mlp_enc.input, outputs=mlp_enc.outputs[0])

    h_cnn = cnn_ext.predict(X1, batch_size=128, verbose=0)
    h_mlp = mlp_ext.predict(X2, batch_size=128, verbose=0)

    print(f"Latent Shapes: CNN={h_cnn.shape}, MLP={h_mlp.shape}")

    # FUSE: [Latent Payload (128) + Latent Stats (128)]
    # This creates a semantic vector of size 256
    X_final = np.concatenate([h_cnn, h_mlp], axis=1)
    print(f"Fused Super-Vector Shape: {X_final.shape}")

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task_head_no_smote(X_final, y_bin, "Binary Task")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))

    # ==========================================
    # EXPERIMENT 2: VPN CATEGORY
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY")
    print("="*40)

    mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final[mask]
    y_cat_raw = df_labels.loc[mask, 'Category']

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)
    y_test, y_pred = train_task_head_no_smote(X_cat, y_cat, "VPN Category")
    print(classification_report(y_test, y_pred, target_names=le_cat.classes_))

    # ==========================================
    # EXPERIMENT 3: VPN TOP APPS
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN TOP 6 APPS")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask = df_labels['Application'].isin(target_apps)
    X_app = X_final[mask]
    y_app_raw = df_labels.loc[mask, 'Application']

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)
    y_test, y_pred = train_task_head_no_smote(X_app, y_app, "VPN Top Apps")
    print(classification_report(y_test, y_pred, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV ---
Columns Found -> V1 (Payload): 128, V2 (Stats): 137
Normalizing Features...
Reconstructing Labels...

--- Generating Latent Representations (CNN + MLP) ---
Loading Weights:
 - /content/drive/MyDrive/1 Skripsi/27jan/HMVCL_Encoder_CNN.weights.h5
 - /content/drive/MyDrive/1 Skripsi/27jan/HMVCL_Encoder_MLP.weights.h5
Latent Shapes: CNN=(12555, 128), MLP=(12555, 128)
Fused Super-Vector Shape: (12555, 256)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 256)
    Train: 1255 | Test: 11300
    >>> Binary Task Weighted F1: 0.9179
              precision    recall  f1-score   support

     Non-VPN       0.95      0.95      0.95      8824
         VPN       0.81      0.82      0.81      2476

    accuracy                           0.92     11300
   macro avg       0.88      0.88      0.88     11300
weighted avg       0.92      0.92      0.92     11300


 EXPERIMENT 2: VPN CATEGORY

>>> Starting Task: VPN Category
    Data Shape: 

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Weights (From your Standard Pre-training)
# Update exact filenames if they differ on your drive
CNN_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_CNN.weights.h5")
MLP_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_MLP.weights.h5")

# Config
LABEL_PERCENTAGE = 0.20
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Architecture Definitions (Must match Pre-training) ---

# A. CNN Encoder (View 1)
def get_cnn_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((input_dim, 1))(inputs)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_Encoder")

# B. MLP Encoder (View 2)
def get_mlp_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="MLP_Encoder")

# --- 2. Data Loading Helper ---
def load_and_prep_data():
    print("--- Loading Merged CSV ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # View 1: Alpha
    cols_v1 = sorted([c for c in df.columns if c.startswith('alpha_alpha_pp_')],
                     key=lambda x: int(x.split('_')[-1]))

    # View 2: Stats (Beta + Gamma + FFT)
    exclude = ['application', 'category', 'binary_type', 'filename']
    cols_v2 = [c for c in df.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
               and not any(x in c for x in exclude)]

    print(f"Columns Found -> V1 (Payload): {len(cols_v1)}, V2 (Stats): {len(cols_v2)}")

    X1 = df[cols_v1].values.astype('float32')
    X2 = df[cols_v2].values.astype('float32')

    # Normalize Stats (Essential for MLP)
    print("Normalizing Features...")
    X2 = StandardScaler().fit_transform(X2)

    return df, X1, X2

# --- 3. Trainer Helper (No SMOTE) ---
def train_task_head_no_smote(X_subset, y_subset, task_name):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # Use sample weights for balance
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")
    return y_test, y_pred

# --- 4. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X1, X2 = load_and_prep_data()
    if df_raw is None: return

    # 2. Reconstruct Labels
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # 3. Feature Fusion (Dual Latent Representations)
    print("\n--- Generating Latent Representations (CNN + MLP) ---")

    # Define Models
    cnn_enc = get_cnn_encoder(X1.shape[1])
    mlp_enc = get_mlp_encoder(X2.shape[1])

    print(f"Loading Weights:\n - {CNN_WEIGHTS_PATH}\n - {MLP_WEIGHTS_PATH}")
    try:
        cnn_enc.load_weights(CNN_WEIGHTS_PATH)
        mlp_enc.load_weights(MLP_WEIGHTS_PATH)
    except Exception as e:
        print(f"FATAL: Could not load weights.\n{e}")
        return

    # Extract Latent Features (Cut off projection head)
    # We use index 0 -> h (representation)
    cnn_ext = Model(inputs=cnn_enc.input, outputs=cnn_enc.outputs[0])
    mlp_ext = Model(inputs=mlp_enc.input, outputs=mlp_enc.outputs[0])

    h_cnn = cnn_ext.predict(X1, batch_size=128, verbose=0)
    h_mlp = mlp_ext.predict(X2, batch_size=128, verbose=0)

    print(f"Latent Shapes: CNN={h_cnn.shape}, MLP={h_mlp.shape}")

    # FUSE: [Latent Payload (128) + Latent Stats (128)]
    # This creates a semantic vector of size 256
    X_final = np.concatenate([h_cnn, h_mlp], axis=1)
    print(f"Fused Super-Vector Shape: {X_final.shape}")

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task_head_no_smote(X_final, y_bin, "Binary Task")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))

    # ==========================================
    # EXPERIMENT 2: VPN CATEGORY
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY")
    print("="*40)

    mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final[mask]
    y_cat_raw = df_labels.loc[mask, 'Category']

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)
    y_test, y_pred = train_task_head_no_smote(X_cat, y_cat, "VPN Category")
    print(classification_report(y_test, y_pred, target_names=le_cat.classes_))

    # ==========================================
    # EXPERIMENT 3: VPN TOP APPS
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN TOP 6 APPS")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask = df_labels['Application'].isin(target_apps)
    X_app = X_final[mask]
    y_app_raw = df_labels.loc[mask, 'Application']

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)
    y_test, y_pred = train_task_head_no_smote(X_app, y_app, "VPN Top Apps")
    print(classification_report(y_test, y_pred, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV ---
Columns Found -> V1 (Payload): 128, V2 (Stats): 137
Normalizing Features...
Reconstructing Labels...

--- Generating Latent Representations (CNN + MLP) ---
Loading Weights:
 - /content/drive/MyDrive/1 Skripsi/27jan/HMVCL_Encoder_CNN.weights.h5
 - /content/drive/MyDrive/1 Skripsi/27jan/HMVCL_Encoder_MLP.weights.h5
Latent Shapes: CNN=(12555, 128), MLP=(12555, 128)
Fused Super-Vector Shape: (12555, 256)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 256)
    Train: 2511 | Test: 10044
    >>> Binary Task Weighted F1: 0.9326
              precision    recall  f1-score   support

     Non-VPN       0.96      0.95      0.96      7843
         VPN       0.82      0.88      0.85      2201

    accuracy                           0.93     10044
   macro avg       0.89      0.91      0.90     10044
weighted avg       0.93      0.93      0.93     10044


 EXPERIMENT 2: VPN CATEGORY

>>> Starting Task: VPN Category
    Data Shape: 

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Weights (From your Standard Pre-training)
# Update exact filenames if they differ on your drive
CNN_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_CNN.weights.h5")
MLP_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_MLP.weights.h5")

# Config
LABEL_PERCENTAGE = 0.30
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Architecture Definitions (Must match Pre-training) ---

# A. CNN Encoder (View 1)
def get_cnn_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((input_dim, 1))(inputs)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_Encoder")

# B. MLP Encoder (View 2)
def get_mlp_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="MLP_Encoder")

# --- 2. Data Loading Helper ---
def load_and_prep_data():
    print("--- Loading Merged CSV ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # View 1: Alpha
    cols_v1 = sorted([c for c in df.columns if c.startswith('alpha_alpha_pp_')],
                     key=lambda x: int(x.split('_')[-1]))

    # View 2: Stats (Beta + Gamma + FFT)
    exclude = ['application', 'category', 'binary_type', 'filename']
    cols_v2 = [c for c in df.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
               and not any(x in c for x in exclude)]

    print(f"Columns Found -> V1 (Payload): {len(cols_v1)}, V2 (Stats): {len(cols_v2)}")

    X1 = df[cols_v1].values.astype('float32')
    X2 = df[cols_v2].values.astype('float32')

    # Normalize Stats (Essential for MLP)
    print("Normalizing Features...")
    X2 = StandardScaler().fit_transform(X2)

    return df, X1, X2

# --- 3. Trainer Helper (No SMOTE) ---
def train_task_head_no_smote(X_subset, y_subset, task_name):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # Use sample weights for balance
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")
    return y_test, y_pred

# --- 4. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X1, X2 = load_and_prep_data()
    if df_raw is None: return

    # 2. Reconstruct Labels
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # 3. Feature Fusion (Dual Latent Representations)
    print("\n--- Generating Latent Representations (CNN + MLP) ---")

    # Define Models
    cnn_enc = get_cnn_encoder(X1.shape[1])
    mlp_enc = get_mlp_encoder(X2.shape[1])

    print(f"Loading Weights:\n - {CNN_WEIGHTS_PATH}\n - {MLP_WEIGHTS_PATH}")
    try:
        cnn_enc.load_weights(CNN_WEIGHTS_PATH)
        mlp_enc.load_weights(MLP_WEIGHTS_PATH)
    except Exception as e:
        print(f"FATAL: Could not load weights.\n{e}")
        return

    # Extract Latent Features (Cut off projection head)
    # We use index 0 -> h (representation)
    cnn_ext = Model(inputs=cnn_enc.input, outputs=cnn_enc.outputs[0])
    mlp_ext = Model(inputs=mlp_enc.input, outputs=mlp_enc.outputs[0])

    h_cnn = cnn_ext.predict(X1, batch_size=128, verbose=0)
    h_mlp = mlp_ext.predict(X2, batch_size=128, verbose=0)

    print(f"Latent Shapes: CNN={h_cnn.shape}, MLP={h_mlp.shape}")

    # FUSE: [Latent Payload (128) + Latent Stats (128)]
    # This creates a semantic vector of size 256
    X_final = np.concatenate([h_cnn, h_mlp], axis=1)
    print(f"Fused Super-Vector Shape: {X_final.shape}")

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task_head_no_smote(X_final, y_bin, "Binary Task")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))

    # ==========================================
    # EXPERIMENT 2: VPN CATEGORY
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY")
    print("="*40)

    mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final[mask]
    y_cat_raw = df_labels.loc[mask, 'Category']

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)
    y_test, y_pred = train_task_head_no_smote(X_cat, y_cat, "VPN Category")
    print(classification_report(y_test, y_pred, target_names=le_cat.classes_))

    # ==========================================
    # EXPERIMENT 3: VPN TOP APPS
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN TOP 6 APPS")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask = df_labels['Application'].isin(target_apps)
    X_app = X_final[mask]
    y_app_raw = df_labels.loc[mask, 'Application']

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)
    y_test, y_pred = train_task_head_no_smote(X_app, y_app, "VPN Top Apps")
    print(classification_report(y_test, y_pred, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV ---
Columns Found -> V1 (Payload): 128, V2 (Stats): 137
Normalizing Features...
Reconstructing Labels...

--- Generating Latent Representations (CNN + MLP) ---
Loading Weights:
 - /content/drive/MyDrive/1 Skripsi/27jan/HMVCL_Encoder_CNN.weights.h5
 - /content/drive/MyDrive/1 Skripsi/27jan/HMVCL_Encoder_MLP.weights.h5
Latent Shapes: CNN=(12555, 128), MLP=(12555, 128)
Fused Super-Vector Shape: (12555, 256)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 256)
    Train: 3766 | Test: 8789
    >>> Binary Task Weighted F1: 0.9358
              precision    recall  f1-score   support

     Non-VPN       0.97      0.95      0.96      6863
         VPN       0.83      0.88      0.86      1926

    accuracy                           0.94      8789
   macro avg       0.90      0.92      0.91      8789
weighted avg       0.94      0.94      0.94      8789


 EXPERIMENT 2: VPN CATEGORY

>>> Starting Task: VPN Category
    Data Shape: (